[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/BMED365-2026/blob/main/Lab3-GenAI-LLM/notebooks/09-chatgpt-claude-api.ipynb)

# ChatGPT and Claude API - Practical Integration

**ELMED219 / BMED365 - Lab 3**

Last updated:<br>
2026-01-03, A. Lundervold

---

> **Note**: This notebook is optional and requires API keys from OpenAI and/or Anthropic. It is intended for students who want to experiment with programmatic use of LLMs.

## Learning Objectives

After this notebook, you should be able to:
- Set up and use OpenAI (ChatGPT) and Anthropic (Claude) APIs
- Implement error handling and rate limiting
- Build a simple medical assistant function
- Understand best practices for API usage in medical contexts

## Contents

1. [Setup and configuration](#1-setup-and-configuration)
2. [OpenAI API (ChatGPT)](#2-openai-api-chatgpt)
3. [Anthropic API (Claude)](#3-anthropic-api-claude)
4. [Error handling](#4-error-handling)
5. [Practical examples](#5-practical-examples)
6. [Security considerations](#6-security-considerations)

---

## 1. Setup and configuration

### Prerequisites

1. API key from [OpenAI](https://platform.openai.com/api-keys) and/or [Anthropic](https://console.anthropic.com/)
2. Python packages: `openai`, `anthropic`

### Best practices for API keys

**Never** hardcode API keys in code. Use environment variables:

```bash
# In terminal:
export OPENAI_API_KEY="sk-..."
export ANTHROPIC_API_KEY="sk-ant-..."
```

Or a `.env` file (which should NOT be committed to git).

### How to set up API keys

If you see "Not configured" for either API, follow these steps:

#### Getting your API keys

1. **OpenAI (ChatGPT)**: 
   - Visit [OpenAI API Keys](https://platform.openai.com/api-keys)
   - Sign up or log in
   - Create a new API key
   - Copy the key (starts with `sk-`)

2. **Anthropic (Claude)**:
   - Visit [Anthropic Console](https://console.anthropic.com/)
   - Sign up or log in
   - Navigate to "API Keys"
   - Create a new key
   - Copy the key (starts with `sk-ant-`)

#### Setting up the keys (choose one method)

**Method 1: Using a `.env` file (Recommended)**

1. Create a `.env` file in the same directory as this notebook (or in the project root)
2. Add your keys:
   ```bash
   OPENAI_API_KEY=sk-your-key-here
   ANTHROPIC_API_KEY=sk-ant-your-key-here
   ```
3. **Important**: Make sure `.env` is in your `.gitignore` file (never commit API keys!)
4. Re-run the setup cell above - the notebook will automatically load keys from `.env`

**Method 2: Terminal environment variables**

```bash
export OPENAI_API_KEY="sk-your-key-here"
export ANTHROPIC_API_KEY="sk-ant-your-key-here"
```

**Method 3: Google Colab**

If running in Colab, you can set environment variables in a code cell:
```python
import os
os.environ["OPENAI_API_KEY"] = "sk-your-key-here"
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-your-key-here"
```

After setting up your keys, **re-run the setup cell** above. You should see:
```
API availability:
  OpenAI (ChatGPT): Available
  Anthropic (Claude): Available
```

If keys are not configured, the notebook will use simulated responses for demonstration purposes.

In [1]:
# Install necessary packages (run if needed)
# !pip install openai anthropic python-dotenv

import os
import sys

# Check if we are in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    !pip install openai anthropic python-dotenv --quiet

# Load environment variables from .env if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Environment variables loaded from .env")
except ImportError:
    print("python-dotenv not installed - using existing environment variables")

# Check available APIs
OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
ANTHROPIC_AVAILABLE = bool(os.getenv("ANTHROPIC_API_KEY"))

print(f"\nAPI availability:")
print(f"  OpenAI (ChatGPT): {'Available' if OPENAI_AVAILABLE else 'Not configured'}")
print(f"  Anthropic (Claude): {'Available' if ANTHROPIC_AVAILABLE else 'Not configured'}")

if not (OPENAI_AVAILABLE or ANTHROPIC_AVAILABLE):
    print("\n⚠️  No API keys found.")
    print("   Examples will show simulated responses.")

Environment variables loaded from .env

API availability:
  OpenAI (ChatGPT): Available
  Anthropic (Claude): Available


---

## 2. OpenAI API (ChatGPT)

In [2]:
from typing import Optional

def chat_with_gpt(prompt: str, 
                  model: str = "gpt-3.5-turbo",
                  temperature: float = 0.7,
                  system_prompt: str = None) -> str:
    """
    Send a message to ChatGPT and get a response.
    
    Args:
        prompt: User's message
        model: GPT model (gpt-3.5-turbo, gpt-4, etc.)
        temperature: 0-2, where 0 is deterministic
        system_prompt: Optional system instruction
    
    Returns:
        Model's response
    """
    if not OPENAI_AVAILABLE:
        return f"[SIMULATED RESPONSE]\nThis is a simulated response for: {prompt[:100]}..."
    
    try:
        from openai import OpenAI
        client = OpenAI()
        
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"Error in API call: {e}"

# Test
response = chat_with_gpt(
    "Briefly explain what HbA1c measures.",
    temperature=0.3,
    system_prompt="You are a helpful medical assistant. Answer briefly and precisely."
)
print("Response:")
print(response)

Response:
HbA1c measures the average blood sugar levels over the past 2-3 months.


---

## 3. Anthropic API (Claude)

The chat_with_claude() function now uses a current model. When you re-run the cell, it should work with your Anthropic API key.<br>
Note: The function still accepts any model name as a parameter, so you can also use:<br>

- claude-opus-4-1-20250805 (most powerful)
- claude-sonnet-4-5-20250929 (fastest, most cost-effective)
- Or any other current model from Anthropic's API

Se: https://platform.claude.com/docs/en/api/overview

In [8]:
# Claude API pricing (as of January 2026)
# Standard pricing for prompts ≤ 200K tokens
CLAUDE_PRICING = {
    "claude-sonnet-4-5-20250929": {
        "input_per_million": 3.00,
        "output_per_million": 15.00
    },
    "claude-opus-4-5-20251101": {
        "input_per_million": 15.00,
        "output_per_million": 75.00
    },
    "claude-haiku-4-5-20251001": {
        "input_per_million": 0.80,
        "output_per_million": 4.00
    },
    # Default pricing for other models (Sonnet pricing)
    "default": {
        "input_per_million": 3.00,
        "output_per_million": 15.00
    }
}

def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> dict:
    """
    Calculate the cost of an API call based on model and token usage.
    
    Args:
        model: Model name
        input_tokens: Number of input tokens
        output_tokens: Number of output tokens
    
    Returns:
        Dict with 'input_cost', 'output_cost', and 'total_cost'
    """
    # Get pricing for the model, or use default
    pricing = CLAUDE_PRICING.get(model, CLAUDE_PRICING["default"])
    
    input_cost = (input_tokens / 1_000_000) * pricing["input_per_million"]
    output_cost = (output_tokens / 1_000_000) * pricing["output_per_million"]
    total_cost = input_cost + output_cost
    
    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

def chat_with_claude(prompt: str,
                     model: str = "claude-sonnet-4-5-20250929",
                     temperature: float = 0.7,
                     system_prompt: str = None) -> dict:
    """
    Send a message to Claude and get a response.
    
    Args:
        prompt: User's message
        model: Claude model (claude-3-sonnet, claude-3-opus, etc.)
        temperature: 0-1
        system_prompt: Optional system instruction
    
    Returns:
        Dict with 'text', 'model', 'input_tokens', 'output_tokens', 'total_tokens',
        'input_cost', 'output_cost', and 'total_cost'
    """
    if not ANTHROPIC_AVAILABLE:
        return {
            "text": f"[SIMULATED RESPONSE]\nThis is a simulated response for: {prompt[:100]}...",
            "model": model,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "input_cost": 0.0,
            "output_cost": 0.0,
            "total_cost": 0.0
        }
    
    try:
        from anthropic import Anthropic
        client = Anthropic()
        
        kwargs = {
            "model": model,
            "max_tokens": 1024,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]
        }
        
        if system_prompt:
            kwargs["system"] = system_prompt
        
        response = client.messages.create(**kwargs)
        
        # Get token usage
        input_tokens = response.usage.input_tokens
        output_tokens = response.usage.output_tokens
        total_tokens = input_tokens + output_tokens
        
        # Calculate cost
        cost_info = calculate_cost(response.model, input_tokens, output_tokens)
        
        # Return response text, model, token usage, and cost
        return {
            "text": response.content[0].text,
            "model": response.model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "input_cost": cost_info["input_cost"],
            "output_cost": cost_info["output_cost"],
            "total_cost": cost_info["total_cost"]
        }
        
    except Exception as e:
        return {"error": f"Error in API call: {e}"}

# Test
result = chat_with_claude(
    "What is the difference between Type 1 and Type 2 diabetes?",
    temperature=0.3,
    system_prompt="You are a medical assistant. Explain in a patient-friendly way."
)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    print(f"Model: {result['model']}")
    print(f"Input tokens: {result['input_tokens']}")
    print(f"Output tokens: {result['output_tokens']}")
    print(f"Total tokens: {result['total_tokens']}")
    print(f"\nCost breakdown:")
    print(f"  Input cost:  ${result['input_cost']:.6f}")
    print(f"  Output cost: ${result['output_cost']:.6f}")
    print(f"  Total cost:  ${result['total_cost']:.6f} (${result['total_cost']*100:.2f} cents)")
    print("\nResponse:")
    print(result['text'])


Model: claude-sonnet-4-5-20250929
Input tokens: 38
Output tokens: 321
Total tokens: 359

Cost breakdown:
  Input cost:  $0.000114
  Output cost: $0.004815
  Total cost:  $0.004929 ($0.49 cents)

Response:
# Type 1 vs Type 2 Diabetes

Great question! Both are forms of diabetes, but they develop differently:

## **Type 1 Diabetes**
- **What happens**: Your body's immune system attacks the cells in your pancreas that make insulin, so you produce little to no insulin
- **When it starts**: Usually in childhood or young adulthood (though it can occur at any age)
- **Cause**: Autoimmune condition - not caused by lifestyle
- **Treatment**: Requires insulin injections or a pump daily
- **Prevalence**: About 5-10% of diabetes cases

## **Type 2 Diabetes**
- **What happens**: Your body either doesn't use insulin properly (insulin resistance) or doesn't make enough insulin over time
- **When it starts**: Usually in adults, but increasingly seen in younger people
- **Cause**: Combination of genetic

---

## 4. Error handling

When using APIs programmatically, robust error handling is critical.

In [9]:
import time

class RobustAPIClient:
    """
    Wrapper with retry logic and rate limiting.
    """
    
    def __init__(self, max_retries: int = 3, base_delay: float = 1.0):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.last_call_time = 0
        self.min_interval = 0.5  # Minimum time between calls (seconds)
    
    def _wait_if_needed(self):
        """Respect rate limits."""
        elapsed = time.time() - self.last_call_time
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call_time = time.time()
    
    def call_api(self, func, *args, **kwargs):
        """
        Execute API call with retry logic.
        """
        for attempt in range(self.max_retries):
            try:
                self._wait_if_needed()
                return func(*args, **kwargs)
                
            except Exception as e:
                error_msg = str(e)
                
                # Check if it's a rate limit error
                if "rate_limit" in error_msg.lower() or "429" in error_msg:
                    wait_time = self.base_delay * (2 ** attempt)
                    print(f"Rate limit - waiting {wait_time}s before retry {attempt + 1}/{self.max_retries}")
                    time.sleep(wait_time)
                else:
                    # Other errors - return error message
                    return f"Error after {attempt + 1} attempts: {e}"
        
        return "Maximum number of attempts reached"

# Example usage
print("RobustAPIClient ready for use.")
print("This class handles rate limits and network errors automatically.")

RobustAPIClient ready for use.
This class handles rate limits and network errors automatically.


---

## 5. Practical examples

### Example: Medical assistant function

In [10]:
def medical_assistant(question: str, context: str = None) -> dict:
    """
    Medical assistant with safety instructions.
    
    Returns:
        Dict with 'answer' and 'warnings'
    """
    
    system_prompt = """You are a medical information assistant.

IMPORTANT RULES:
1. Provide only general health information
2. Never give specific diagnoses or treatment recommendations
3. Always recommend consulting a doctor for personal health concerns
4. Be clear about limitations in your knowledge
5. For acute symptoms: Recommend immediately contacting healthcare personnel

Answer in English, briefly and precisely."""
    
    full_prompt = question
    if context:
        full_prompt = f"Context: {context}\n\nQuestion: {question}"
    
    # Here we would normally call the API
    # For demonstration, we simulate the response
    
    simulated_answer = {
        "answer": f"[Simulated answer for: {question[:50]}...]\n\n" + 
                "This is general health information. " +
                "For personal assessment, contact healthcare personnel.",
        "warnings": [
            "This information does not replace medical advice",
            "Consult a doctor for personal assessment"
        ]
    }
    
    return simulated_answer

# Test
result = medical_assistant(
    "What are common side effects of metformin?",
    context="Patient with newly diagnosed type 2 diabetes"
)

print("MEDICAL ASSISTANT")
print("=" * 50)
print(f"\nAnswer:\n{result['answer']}")
print(f"\nWarnings:")
for warning in result['warnings']:
    print(f"  ⚠️  {warning}")

MEDICAL ASSISTANT

Answer:
[Simulated answer for: What are common side effects of metformin?...]

This is general health information. For personal assessment, contact healthcare personnel.

Warnings:
  ⚠️  This information does not replace medical advice
  ⚠️  Consult a doctor for personal assessment


---

## 6. Security considerations

### When using LLM APIs in healthcare contexts:

**1. Privacy**
- Never send real patient data to external APIs without approval
- Consider local models (Ollama, etc.) for sensitive data
- Check data processing agreements with API provider

**2. Hallucination**
- LLMs can make up facts
- Always verify medical information
- Use low temperature for fact-based questions

**3. Logging**
- Log all API calls for traceability
- Delete logs with patient information according to policy

**4. Human oversight**
- AI output should always be reviewed by healthcare personnel
- Implement clear warnings in user interface

---

## Summary

### Key points

1. **API keys** should never be hardcoded - use environment variables
2. **Error handling** with retry logic is essential
3. **System prompts** with safety instructions are critical for medical use
4. **Privacy** must be safeguarded - consider local models for sensitive data
5. **Human oversight** is always required for medical AI

### Resources

- [OpenAI API documentation](https://platform.openai.com/docs)
- [Anthropic API documentation](https://docs.anthropic.com)
- [OpenAI Cookbook](https://cookbook.openai.com/)

---

*Back to [Lab 3 overview](../README.md)*